In [1]:
"""
Bank Personal Loan Analysis
Step 1: Data Loading, Cleaning, Validation & EDA
"""

import pandas as pd
import numpy as np

# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
df = pd.read_csv(r"C:\Users\shrey\Desktop\projects\GITHUB_PROJECTS\Bank Loan Report\Bank_Personal_Loan_Modelling(1).csv")

print("=" * 60)
print("STEP 1: DATA LOADING")
print("=" * 60)
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumns:\n{list(df.columns)}")
print(f"\nData Types:\n{df.dtypes}")

# ─────────────────────────────────────────────
# 2. DATA VALIDATION
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: DATA VALIDATION")
print("=" * 60)

# Null values
print("\nNull Values:")
print(df.isnull().sum())

# Duplicate rows
dupes = df.duplicated().sum()
print(f"\nDuplicate Rows: {dupes}")

# Data type issue: CCAvg stored as string "1/60" instead of float "1.60"
print(f"\nCCAvg sample (before fix): {df['CCAvg'].head(5).tolist()}")

# Negative experience check
neg_exp = (df['Experience'] < 0).sum()
print(f"Negative Experience values: {neg_exp}")

# ─────────────────────────────────────────────
# 3. DATA CLEANING
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: DATA CLEANING")
print("=" * 60)

# Fix CCAvg: replace "/" with "." and cast to float
df['CCAvg'] = df['CCAvg'].str.replace('/', '.').astype(float)
print(f"CCAvg fixed. Sample: {df['CCAvg'].head(5).tolist()}")

# Normalize column names (remove spaces)
df.columns = df.columns.str.strip().str.replace(' ', '_')
print(f"\nCleaned column names:\n{list(df.columns)}")

# Fix negative Experience (clamp to 0)
df['Experience'] = df['Experience'].clip(lower=0)
neg_after = (df['Experience'] < 0).sum()
print(f"Negative Experience after fix: {neg_after}")

# Drop ID and ZIP Code (not predictive)
df.drop(columns=['ID', 'ZIP_Code'], inplace=True)
print(f"\nDropped 'ID' and 'ZIP_Code'. New shape: {df.shape}")

# ─────────────────────────────────────────────
# 4. FEATURE ENGINEERING
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: FEATURE ENGINEERING")
print("=" * 60)

# CC spend to income ratio
df['CCAvg_to_Income'] = (df['CCAvg'] / df['Income']).round(4)

# Income groups
df['Income_Group'] = pd.cut(
    df['Income'],
    bins=[0, 40, 80, 130, 250],
    labels=['Low', 'Medium', 'High', 'Very_High']
)

print("New features added: CCAvg_to_Income, Income_Group")
print(f"\nIncome_Group distribution:\n{df['Income_Group'].value_counts()}")

# ─────────────────────────────────────────────
# 5. EXPLORATORY DATA ANALYSIS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: EXPLORATORY DATA ANALYSIS")
print("=" * 60)

# Summary statistics
print("\nSummary Statistics:")
print(df.describe().round(2).to_string())

# Target distribution
total = len(df)
approved = df['Personal_Loan'].sum()
rejected = total - approved
print(f"\nTarget Variable (Personal_Loan):")
print(f"  Approved (1): {approved:,}  ({approved/total*100:.1f}%)")
print(f"  Rejected (0): {rejected:,}  ({rejected/total*100:.1f}%)")

# Approval rate by key segments
print("\nApproval Rate by Education Level:")
edu_map = {1: 'Undergrad', 2: 'Graduate', 3: 'Advanced'}
edu_agg = df.groupby('Education')['Personal_Loan'].agg(['mean', 'count'])
edu_agg.index = edu_agg.index.map(edu_map)
edu_agg.columns = ['Approval Rate', 'Count']
edu_agg['Approval Rate'] = (edu_agg['Approval Rate'] * 100).round(1).astype(str) + '%'
print(edu_agg.to_string())

print("\nApproval Rate by Income Group:")
inc_agg = df.groupby('Income_Group', observed=True)['Personal_Loan'].agg(['mean', 'count'])
inc_agg.columns = ['Approval Rate', 'Count']
inc_agg['Approval Rate'] = (inc_agg['Approval Rate'] * 100).round(1).astype(str) + '%'
print(inc_agg.to_string())

print("\nApproval Rate by CD Account:")
cd_agg = df.groupby('CD_Account')['Personal_Loan'].agg(['mean', 'count'])
cd_agg.index = cd_agg.index.map({0: 'No CD Account', 1: 'Has CD Account'})
cd_agg.columns = ['Approval Rate', 'Count']
cd_agg['Approval Rate'] = (cd_agg['Approval Rate'] * 100).round(1).astype(str) + '%'
print(cd_agg.to_string())

print("\nApproval Rate by Family Size:")
fam_agg = df.groupby('Family')['Personal_Loan'].agg(['mean', 'count'])
fam_agg.columns = ['Approval Rate', 'Count']
fam_agg['Approval Rate'] = (fam_agg['Approval Rate'] * 100).round(1).astype(str) + '%'
print(fam_agg.to_string())

print("\nAvg CCAvg (monthly credit card spend):")
ccavg_agg = df.groupby('Personal_Loan')['CCAvg'].mean().round(2)
ccavg_agg.index = ccavg_agg.index.map({0: 'Rejected', 1: 'Approved'})
print(ccavg_agg.to_string())

print("\nAvg Income:")
income_agg = df.groupby('Personal_Loan')['Income'].agg(['mean', 'min', 'max']).round(1)
income_agg.index = income_agg.index.map({0: 'Rejected', 1: 'Approved'})
print(income_agg.to_string())

# Correlation with target
print("\nCorrelation with Personal_Loan (numeric features):")
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()['Personal_Loan'].drop('Personal_Loan').sort_values(ascending=False)
print(corr.round(3).to_string())

# ─────────────────────────────────────────────
# 6. SAVE CLEANED DATA
# ─────────────────────────────────────────────
df.to_csv("cleaned_loan_data.csv", index=False)
print("\n" + "=" * 60)
print("Cleaned dataset saved to: cleaned_loan_data.csv")
print("=" * 60)


STEP 1: DATA LOADING
Shape: 5000 rows × 14 columns

Columns:
['ID', 'Age', 'Experience', 'Income', 'ZIP Code', 'Family', 'CCAvg', 'Education', 'Mortgage', 'Personal Loan', 'Securities Account', 'CD Account', 'Online', 'CreditCard']

Data Types:
ID                     int64
Age                    int64
Experience             int64
Income                 int64
ZIP Code               int64
Family                 int64
CCAvg                 object
Education              int64
Mortgage               int64
Personal Loan          int64
Securities Account     int64
CD Account             int64
Online                 int64
CreditCard             int64
dtype: object

STEP 2: DATA VALIDATION

Null Values:
ID                    0
Age                   0
Experience            0
Income                0
ZIP Code              0
Family                0
CCAvg                 0
Education             0
Mortgage              0
Personal Loan         0
Securities Account    0
CD Account            0
Online  